# 👁️ DRISHTI AI - Deep Learning Model Training on Cloud GPU (Google Colab)
This notebook trains the **Drishti AI** Diabetic Retinopathy classification backbone (**EfficientNet-B3 / ConvNeXt-Base**) on **APTOS 2019 Blindness Detection** dataset with **Quadratic Weighted Kappa (QWK) Optimization**.

### ⚡ Hardware Requirement:
Go to **Runtime > Change runtime type > Hardware accelerator: T4 GPU** (or A100 if Colab Pro).

In [ ]:
# Step 1: Verify GPU
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

### Step 2: Clone Repository or Mount Google Drive
Choose either to clone from your GitHub repository or mount your Google Drive if your files are stored there.

In [ ]:
# Option A: Clone from GitHub (Replace with your repository URL)
# !git clone https://github.com/YOUR_USERNAME/DrishtiAI.git
# %cd DrishtiAI

# Option B: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# If DrishtiAI is in Drive, cd into it:
# %cd /content/drive/MyDrive/DrishtiAI

### Step 3: Install Required Dependencies
Installs PyTorch, Albumentations, timm, and clinical vision libraries.

In [ ]:
!pip install --quiet timm albumentations opencv-python-headless scikit-learn pandas pillow tqdm kaggle

### Step 4: Download APTOS 2019 Dataset from Kaggle
Upload your `kaggle.json` API token or enter credentials below to download the official dataset (~3,662 high-res fundus images).

In [ ]:
import os
import getpass

# Upload kaggle.json or input credentials
os.environ['KAGGLE_USERNAME'] = input("Enter your Kaggle Username: ")
os.environ['KAGGLE_KEY'] = getpass.getpass("Enter your Kaggle API Key: ")

# Create data directory
!mkdir -p data/aptos2019
# Download competition dataset
!kaggle competitions download -c aptos2019-blindness-detection -p data/aptos2019
!unzip -q -o data/aptos2019/aptos2019-blindness-detection.zip -d data/aptos2019/
print("✅ Dataset downloaded and extracted successfully!")

### Step 5: Start Model Training with Cloud GPU Acceleration
- Uses **Automatic Mixed Precision (AMP)** for 2.5x faster throughput.
- Optimizes **Quadratic Weighted Kappa (QWK)** with **Hybrid QWK-CrossEntropy Loss**.
- Uses Cosine Annealing learning rate schedule.

In [ ]:
# Run training (Target Fold 1 for fast single fold, or omit --target_fold for full 5-fold CV)
!python ai/train.py \
    --csv data/aptos2019/train.csv \
    --images_dir data/aptos2019/train_images \
    --backbone efficientnet_b3 \
    --epochs 15 \
    --batch_size 16 \
    --lr 1e-4 \
    --target_fold 1 \
    --num_workers 2 \
    --output_dir artifacts/models

### Step 6: Export & Download Trained Model Checkpoint
Copy the trained `drishti_best_model.pth` back to your local machine or Google Drive.

In [ ]:
from google.colab import files
import os

best_model_path = "artifacts/models/drishti_best_model.pth"
if os.path.exists(best_model_path):
    print(f"💾 Best model found ({os.path.getsize(best_model_path) / 1e6:.1f} MB). Downloading...")
    files.download(best_model_path)
else:
    print("Model checkpoint not found. Check training output.")